# For each GP, check average number of genes per cell?
&rarr; from this can filter GP with low number of genes

In [1]:
from geneformer import ENSEMBL_DICTIONARY_FILE
import pandas as pd
import numpy as np
import scanpy as sc

In [2]:
name_to_ens = pd.read_pickle(ENSEMBL_DICTIONARY_FILE)

In [3]:
gpdb = pd.read_csv('gp_nw_all.csv')

In [4]:
gpdb_ens = {}

for gp in gpdb.columns:
    gpdb_ens[gp] = [name_to_ens[g] if not pd.isna(g) and g in name_to_ens else np.nan for g in gpdb[gp].values]
    
gpdb_ens = pd.DataFrame(gpdb_ens)

In [5]:
#cd34 = sc.read_h5ad('data/processed/cd34_fixed.h5ad')
mnc = sc.read_h5ad('data/processed/mnc.h5ad')

In [6]:
cd34 = sc.read_h5ad('/nfs/team361/cytomeister/data/hspc/CD34_RNA_98266cells_to_share.h5ad')
cd34.var = cd34.var.reset_index()
cd34.var['ensembl_id'] = cd34.var['gene_ids']
cd34.var = cd34.var.set_index('gene_ids')

In [7]:
cd34

AnnData object with n_obs × n_vars = 98266 × 36601
    obs: 'runid_mrna_sample', 'assignment_id', 'age', 'mrna_samples', 'runid', 'sorting', 'runid_prot_samples', 'prot_samples', 'biological_replicate_labID', 'sex', 'tissue', 'age_general', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'phase', 'S_score', 'G2M_score', 'celltype_v1', 'leiden', 'celltype_v2', 'donor_tissue'
    var: 'index', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches', 'mean', 'std', 'ensembl_id'
    uns: 'celltype_v1_colors', 'celltype_v2_colors', 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_rna_umap', 'X_umap'
    varm: 'PCs'
    layers: 'counts', '

In [8]:
# cd34.X = cd34.layers['counts']

In [9]:
cb = sc.read_h5ad('data/processed/cb.h5ad')

In [10]:
hsc = sc.read_h5ad('data/processed/sakurai_hsc.h5ad')
zeng = sc.read_h5ad('data/processed/zeng.h5ad')
li = sc.read_h5ad('data/processed/li.h5ad')

In [11]:
expr = {
    'cd34' : cd34,
    'mnc' : mnc, 
    'cb' : cb, 
    'hsc' : hsc,
    'zeng' : zeng, 
    'li' : li
}

In [12]:
for gp in gpdb_ens.columns[:5]:
    print(f"Processing gene group: {gp}")  # Print the name of the current gene group
    
    genes = gpdb_ens[gp].dropna().values  # Get the list of non-null gene names in the current group
    
    for name, adata in expr.items():
        print(f"  Processing dataset: {name}")  # Print the name of the current dataset
        
        # Subset adata to only include genes that are in the current list of genes
        bdata = adata[:, adata.var.index.isin(genes)]  
        
        # Calculate the number of genes with non-zero expression in each cell
        non_zero_counts = (bdata.X > 0).sum(axis=1)  # Sum of non-zero values for each row (cell)
        
        # Calculate the average number of genes with non-zero expression per cell
        average_non_zero_genes = non_zero_counts.mean()  
        
        print(f"    Average number of genes with non-zero expression per cell: {average_non_zero_genes:.2f}")
    
    print('')


Processing gene group: GP_USF1
  Processing dataset: cd34
    Average number of genes with non-zero expression per cell: 35.88
  Processing dataset: mnc
    Average number of genes with non-zero expression per cell: 33.39
  Processing dataset: cb
    Average number of genes with non-zero expression per cell: 42.06
  Processing dataset: hsc
    Average number of genes with non-zero expression per cell: 39.72
  Processing dataset: zeng
    Average number of genes with non-zero expression per cell: 21.08
  Processing dataset: li
    Average number of genes with non-zero expression per cell: 26.06

Processing gene group: GP_NFE2L2
  Processing dataset: cd34
    Average number of genes with non-zero expression per cell: 41.35
  Processing dataset: mnc
    Average number of genes with non-zero expression per cell: 41.75
  Processing dataset: cb
    Average number of genes with non-zero expression per cell: 57.47
  Processing dataset: hsc
    Average number of genes with non-zero expression p

In [13]:
gp_to_drop = []

for gp in gpdb_ens.columns:
    print(f"Processing gene group: {gp}") 
    
    genes = gpdb_ens[gp].dropna().values  # Get the list of non-null gene names in the current group
    n_low = 0
    
    for name, adata in expr.items():
        # Subset adata to only include genes that are in the current list of genes
        bdata = adata[:, adata.var.index.isin(genes)]  
        
        # Calculate the number of genes with non-zero expression in each cell
        non_zero_counts = (bdata.X > 0).sum(axis=1)  # Sum of non-zero values for each row (cell)
        
        # Calculate the average number of genes with non-zero expression per cell
        average_non_zero_genes = non_zero_counts.mean()  
        
        if average_non_zero_genes < 10:
            print(f"  Processing dataset: {name}") 
            print(f"    Average number of genes with non-zero expression per cell: {average_non_zero_genes:.2f}")
            print('')
            
        if average_non_zero_genes < 5:
            n_low += 1
            
    
    if n_low > 2:
        gp_to_drop.append(gp)
    


Processing gene group: GP_USF1
Processing gene group: GP_NFE2L2
Processing gene group: GP_RUNX1
Processing gene group: GP_FOXO3
Processing gene group: GP_MYB
Processing gene group: GP_E2F4
Processing gene group: GP_IRF1
Processing gene group: GP_GATA1
Processing gene group: GP_CTCF
Processing gene group: GP_MYCN
Processing gene group: GP_ATF4
Processing gene group: GP_ATF3
Processing gene group: GP_JUNB
  Processing dataset: zeng
    Average number of genes with non-zero expression per cell: 9.06

Processing gene group: GP_JUND
  Processing dataset: zeng
    Average number of genes with non-zero expression per cell: 8.09

  Processing dataset: li
    Average number of genes with non-zero expression per cell: 8.95

Processing gene group: GP_GATA2
  Processing dataset: zeng
    Average number of genes with non-zero expression per cell: 9.10

Processing gene group: GP_DDIT3
Processing gene group: GP_TAL1
  Processing dataset: zeng
    Average number of genes with non-zero expression per c

In [14]:
gp_to_drop = list(set(gp_to_drop))

In [15]:
gp_to_drop

['GP_IRF5', 'GP_NFE2', 'GP_MEIS1', 'GP_KLF1', 'GP_ETV6', 'GP_SOX6', 'GP_NFIX']

In [16]:
gpdb_ens = gpdb_ens.drop(columns = gp_to_drop)

In [17]:
gpdb_ens.head()

,GP_USF1,GP_NFE2L2,GP_RUNX1,GP_FOXO3,GP_MYB,GP_E2F4,GP_IRF1,GP_GATA1,GP_CTCF,GP_MYCN,...,GP_PRDM1,GP_IRF2,GP_NR2C2,GP_ERG,GP_IKZF1,GP_SNAI2,GP_NFYB,GP_HOXA9,GP_ZBTB7A,GP_LMO2
0,ENSG00000158773,ENSG00000116044,ENSG00000159216,ENSG00000118689,ENSG00000118513,ENSG00000205250,ENSG00000125347,ENSG00000102145,ENSG00000102974,ENSG00000134323,...,ENSG00000057657,ENSG00000168310,ENSG00000177463,ENSG00000157554,ENSG00000185811,ENSG00000019549,ENSG00000120837,ENSG00000078399,ENSG00000178951,ENSG00000135363
1,ENSG00000268758,ENSG00000181019,ENSG00000164399,ENSG00000117560,ENSG00000130427,ENSG00000101412,ENSG00000101347,ENSG00000171552,ENSG00000090382,ENSG00000104419,...,ENSG00000136997,ENSG00000101347,ENSG00000110092,ENSG00000108622,ENSG00000128322,ENSG00000039068,ENSG00000179583,ENSG00000118785,ENSG00000171873,ENSG00000170180
2,ENSG00000169710,ENSG00000001084,ENSG00000187010,ENSG00000111276,ENSG00000100284,ENSG00000105173,ENSG00000111537,ENSG00000104870,ENSG00000142192,ENSG00000122641,...,ENSG00000100811,ENSG00000007171,ENSG00000019186,ENSG00000163513,ENSG00000107447,ENSG00000139618,ENSG00000019582,ENSG00000196411,ENSG00000049540,ENSG00000133134
3,ENSG00000254647,ENSG00000160783,ENSG00000133703,ENSG00000142208,ENSG00000172216,ENSG00000170312,ENSG00000007171,ENSG00000211891,ENSG00000136997,ENSG00000136960,...,ENSG00000179583,ENSG00000006071,ENSG00000026508,ENSG00000204248,ENSG00000113441,ENSG00000163347,ENSG00000204592,ENSG00000151247,ENSG00000197894,ENSG00000123405
4,ENSG00000121966,ENSG00000130066,ENSG00000170345,ENSG00000078142,ENSG00000147155,ENSG00000124762,ENSG00000197110,ENSG00000166947,ENSG00000169375,ENSG00000135903,...,ENSG00000196092,ENSG00000108055,ENSG00000101966,ENSG00000100292,ENSG00000114812,ENSG00000125398,ENSG00000143390,ENSG00000115758,ENSG00000105664,ENSG00000179776


In [18]:
gpdb_ens.shape

(227, 31)

In [19]:
gpdb_ens.to_csv('gpdb_nw_ens.csv', index = False)

# Add progeny

In [20]:
progeny = pd.read_csv('gpdb_progeny_200.csv')

In [21]:
progeny_ens = {}

for gp in progeny.columns:
    ens_list = [name_to_ens[g] if not pd.isna(g) and g in name_to_ens else np.nan for g in progeny[gp].values]
    ens_list = ens_list + [np.nan for _ in range(len(gpdb_ens) - len(ens_list))]
    progeny_ens[gp] = ens_list
    
progeny_ens = pd.DataFrame(progeny_ens)

In [22]:
gpdb_tf = pd.concat([gpdb_ens, progeny_ens], axis = 1)

In [23]:
gpdb_tf.to_csv('gpdb_tf.csv', index = False)

In [24]:
gpdb_tf

,GP_USF1,GP_NFE2L2,GP_RUNX1,GP_FOXO3,GP_MYB,GP_E2F4,GP_IRF1,GP_GATA1,GP_CTCF,GP_MYCN,...,JAK-STAT,MAPK,NFkB,p53,PI3K,TGFb,TNFa,Trail,VEGF,WNT
0,ENSG00000158773,ENSG00000116044,ENSG00000159216,ENSG00000118689,ENSG00000118513,ENSG00000205250,ENSG00000125347,ENSG00000102145,ENSG00000102974,ENSG00000134323,...,ENSG00000089127,ENSG00000139318,ENSG00000118503,ENSG00000135423,ENSG00000120215,NaN,ENSG00000118503,ENSG00000070601,NaN,ENSG00000125378
1,ENSG00000268758,ENSG00000181019,ENSG00000164399,ENSG00000117560,ENSG00000130427,ENSG00000101412,ENSG00000101347,ENSG00000171552,ENSG00000090382,ENSG00000104419,...,ENSG00000138642,ENSG00000198369,ENSG00000081041,ENSG00000135679,ENSG00000185664,ENSG00000134198,ENSG00000159110,ENSG00000183242,ENSG00000109265,ENSG00000105492
2,ENSG00000169710,ENSG00000001084,ENSG00000187010,ENSG00000111276,ENSG00000100284,ENSG00000105173,ENSG00000111537,ENSG00000104870,ENSG00000142192,ENSG00000122641,...,ENSG00000111331,ENSG00000136158,ENSG00000115009,ENSG00000196152,ENSG00000223891,ENSG00000101665,ENSG00000164400,NaN,ENSG00000269155,ENSG00000185149
3,ENSG00000254647,ENSG00000160783,ENSG00000133703,ENSG00000142208,ENSG00000172216,ENSG00000170312,ENSG00000007171,ENSG00000211891,ENSG00000136997,ENSG00000136960,...,ENSG00000188313,ENSG00000244405,ENSG00000100906,ENSG00000161513,ENSG00000170271,ENSG00000086991,ENSG00000146232,ENSG00000061492,NaN,ENSG00000119535
4,ENSG00000121966,ENSG00000130066,ENSG00000170345,ENSG00000078142,ENSG00000147155,ENSG00000124762,ENSG00000197110,ENSG00000166947,ENSG00000169375,ENSG00000135903,...,ENSG00000137628,ENSG00000142627,ENSG00000169429,ENSG00000196734,ENSG00000204228,ENSG00000187498,ENSG00000169242,ENSG00000125245,ENSG00000102763,ENSG00000108244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,ENSG00000090447,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223,ENSG00000122852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
224,ENSG00000170909,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
225,ENSG00000196923,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
gpdb_tf.isna().sum()

GP_USF1        0
GP_NFE2L2     23
GP_RUNX1      32
GP_FOXO3      31
GP_MYB        45
GP_E2F4       58
GP_IRF1       58
GP_GATA1      79
GP_CTCF       82
GP_MYCN       87
GP_ATF4       99
GP_ATF3      117
GP_JUNB      118
GP_JUND      119
GP_GATA2     121
GP_DDIT3     126
GP_TAL1      131
GP_FLI1      132
GP_ELF1      147
GP_RUNX3     157
GP_KLF2      160
GP_PRDM1     160
GP_IRF2      165
GP_NR2C2     167
GP_ERG       168
GP_IKZF1     174
GP_SNAI2     178
GP_NFYB      186
GP_HOXA9     187
GP_ZBTB7A    190
GP_LMO2      197
Androgen      28
EGFR          37
Estrogen      27
Hypoxia       31
JAK-STAT      31
MAPK          31
NFkB          29
p53           35
PI3K          54
TGFb          31
TNFa          32
Trail         67
VEGF          39
WNT           29
dtype: int64